In [1]:
#env Gurobi
import numpy as np
from datetime import datetime, timedelta
import sys
from tqdm import tqdm
from multiprocessing import Pool
import os

from Port_Opt_MaxGeneration_EastCoast_EnvConst import SolvePortOpt_MaxGen_LCOE_Iterator

# Technology-Agnostic Portfolio Optimization

This notebook runs the portfolio optimization for **any combination** of technologies.  
Simply populate the design path lists for the technologies you want to evaluate.  
Leave a list empty  to exclude that technology from the run.

**Supported Technologies:** Wind, Wave, Kite, Coaxial, Tidal

**Rules:**
- At least one technology must have design paths
-  and  are only enforced for technologies with designs > 0
- Set  to how many different designs the optimizer can select (e.g., 1 = pick the best single design)
- Set  to the minimum number of devices that MUST be deployed for that tech (0 = optional)

In [2]:
# ======================================================================
# TECHNOLOGY DESIGN PATHS — list the FULL East Coast 2011-2020 tech outputs.
# The RUN CONFIG below narrows them by time range, region, and (for ocean
# current) operating depth. Leave a list empty to exclude that technology.
# ======================================================================
TO = "C:/Users/rmiller9/Documents/East Coast Model/Tech Outputs"

PathWindDesigns = []
PathWindDesigns.append(f"{TO}/Wind/GenPU_ATB_18MW_2030_EastCoast_2011_2020.npz")
PathWindDesigns.append(f"{TO}/Wind/GenPU_ATB_12MW_2030_EastCoast_2011_2020.npz")

PathWaveDesigns = []
PathWaveDesigns.append(f"{TO}/Wave/GenPU_RM3_EastCoast_2011_2020.npz")

PathKiteDesigns = []
# PathKiteDesigns.append(f"{TO}/Current/kite_54kW_0.5-0.75mRFS_2011_2020.npz")

PathCoaxialDesigns = []
# PathCoaxialDesigns.append(f"{TO}/Coaxial/coax_600kW_1.0mRFS_2011_2020.npz")

PathTidalDesigns = []

PathTransmissionDesign = []
PathTransmissionDesign.append(f"{TO}/Transmission/Transmission_1200MW_East Coast.npz")

## Run Configuration

Set the **time range**, the **spatial region** (a named state *or* a custom lat/lon box),
and the **ocean-current operating depth(s)**. The next cell applies these to every design
path above, writing narrowed copies that the optimizer then runs.

- **State** → looks up the latitude band in `State Lat Ranges.txt` (latitude only).
- **Custom box** → lat *and* lon limits (set `SELECT_STATE = None` to use it).
- **Ocean-current depths** → required if any Kite/Coaxial design is included. Each depth
  becomes a **separate design** in the portfolio (e.g. the same kite at 50 m and at 100 m
  are two options the optimizer can choose between). Leave `[]` if no current devices.

In [3]:
# ======================================================================
# RUN CONFIG
# ======================================================================
# --- Time range (inclusive) ---
START_YEAR = 2016
END_YEAR   = 2018

# --- Spatial region: choose ONE ---
SELECT_STATE = None     # a state name inside of "", or None to use CUSTOM_BOX
CUSTOM_BOX   = CUSTOM_BOX = (40.98, 41.30, -71.13, -70.83)                   # (lat_min, lat_max, lon_min, lon_max), used only if SELECT_STATE is None
#CUSTOM_BOX = (40.98, 41.30, -71.13, -70.83)   # Revolution Wind
#CUSTOM_BOX = (36.85, 37.04, -75.31, -75.12)   # Coastal Virginia
#CUSTOM_BOX = (33.37, 33.52, -78.06, -77.73)   # Carolina Long Bay

# --- Ocean-current operating depths (m) ---
# Required if any Kite/Coaxial designs are listed; each depth -> a separate design.
OCEAN_CURRENT_DEPTHS = []            # e.g. [50, 100, 200]  (leave [] if no current devices)

# --- paths ---
STATE_LAT_FILE = "C:/Users/rmiller9/Documents/East Coast Model/State Lat Ranges.txt"
SLICE_TMP_DIR  = "C:/Users/rmiller9/Documents/East Coast Model/Tech Outputs/_sliced"

### Apply the configuration

Narrows each design to the configured time range and region, collapses ocean-current
depth axes to one flat design per selected depth, and swaps the sliced copies into the
design lists. Run this once; the cells below (summary + optimize) then use the sliced designs.

In [4]:
# ======================================================================
# APPLY RUN CONFIG  — slice designs by time / region / depth
# Sliced inputs are written into the run's own folder under Portfolios/,
# so a run's inputs and outputs live together (delete the run folder to clear both).
# ======================================================================
import os, numpy as np

BASE = "C:/Users/rmiller9/Documents/East Coast Model"

def _load_state_ranges(path):
    ns = {}; exec(open(path).read(), ns); return ns["state_lat_ranges"]

if SELECT_STATE is not None:
    _r = _load_state_ranges(STATE_LAT_FILE)[SELECT_STATE]
    LAT_MIN, LAT_MAX, LON_MIN, LON_MAX = _r[0], _r[1], None, None
    REGION_TAG = SELECT_STATE
elif CUSTOM_BOX is not None:
    LAT_MIN, LAT_MAX, LON_MIN, LON_MAX = CUSTOM_BOX
    REGION_TAG = (globals().get("BOX_NAME")
                  or f"Box{LAT_MIN}-{LAT_MAX}".replace(".", "p").replace("-", "m"))
else:
    raise ValueError("Set SELECT_STATE or CUSTOM_BOX.")

# short tech summary for the run id (e.g. W2_V1_K2)
_counts = [("W", len(PathWindDesigns)), ("V", len(PathWaveDesigns)),
           ("K", len(PathKiteDesigns)), ("X", len(PathCoaxialDesigns)), ("T", len(PathTidalDesigns))]
_tech = "-".join(f"{k}{n}" for k, n in _counts if n > 0) or "none"

RUN_ID  = f"{REGION_TAG}_{START_YEAR}_{END_YEAR}_{_tech}"
RUN_DIR = f"{BASE}/Portfolios/{RUN_ID}"          # the run's home folder (inputs + outputs)
INPUT_DIR = f"{RUN_DIR}/inputs"
os.makedirs(INPUT_DIR, exist_ok=True)

_has_current = len(PathKiteDesigns) > 0 or len(PathCoaxialDesigns) > 0
if _has_current and len(OCEAN_CURRENT_DEPTHS) == 0:
    raise ValueError("OCEAN_CURRENT_DEPTHS must be set when Kite or Coaxial designs are included.")

print(f"Run:    {RUN_ID}")
print(f"Region: lat[{LAT_MIN},{LAT_MAX}]" + (f" lon[{LON_MIN},{LON_MAX}]" if LON_MIN is not None else " (lat only)"))
print(f"Years:  {START_YEAR}-{END_YEAR}   |  current depths: {OCEAN_CURRENT_DEPTHS or 'n/a'}")
print(f"Run folder -> {RUN_DIR}\n")

def _clean_core(path):
    core = os.path.basename(path)[:-4]
    for suf in ("_EastCoast_2011_2020", "_2011_2020", "_EastCoast"):
        if core.endswith(suf): return core[:-len(suf)]
    return core

def _slice_design(path, depth_m=None):
    D = np.load(path, allow_pickle=True); keys = D.files
    tl = D["TimeList"]; yrs = np.array([t.year for t in tl])
    tmask = (yrs >= START_YEAR) & (yrs <= END_YEAR)
    ll = D["LatLong"]; n = ll.shape[0]; lat = ll[:, 0]; lon = ll[:, 1]
    smask = (lat >= LAT_MIN) & (lat <= LAT_MAX)
    if LON_MIN is not None: smask &= (lon >= LON_MIN) & (lon <= LON_MAX)
    ep = D["Energy_pu"]; has_depth = (ep.ndim == 3); di = None
    if has_depth:
        depths = np.atleast_1d(D["Depth_m"])
        if depth_m not in depths:
            raise ValueError(f"depth {depth_m} m not in {os.path.basename(path)} (has {list(depths)})")
        di = int(np.where(depths == depth_m)[0][0])
        if "ValidSites" in keys: smask = smask & D["ValidSites"][di]
    sidx = np.flatnonzero(smask)
    if sidx.size == 0:
        raise ValueError(f"0 sites for {os.path.basename(path)} in region/depth — check config.")
    out = {}
    for k in keys:
        a = D[k]
        if k == "TimeList": out[k] = a[tmask]
        elif k == "Depth_m": out[k] = np.float64(depth_m) if has_depth else a
        elif k == "Energy_pu": out[k] = (a[:, di, :] if has_depth else a)[tmask][:, sidx]
        elif k == "RawResource" and hasattr(a, "ndim") and a.ndim >= 2 and a.shape[0] == len(tl):
            out[k] = (a[:, di, :] if a.ndim == 3 else a)[tmask][:, sidx]
        elif has_depth and hasattr(a, "ndim") and a.ndim == 2 and a.shape[0] == len(depths) and a.shape[1] == n:
            out[k] = a[di][sidx]
        elif hasattr(a, "ndim") and a.ndim >= 1 and a.shape[0] == n: out[k] = a[sidx, ...]
        else: out[k] = a
    D.close()
    base = f"{_clean_core(path)}_{REGION_TAG}_{START_YEAR}_{END_YEAR}" + (f"_{int(depth_m)}m" if has_depth else "")
    op = f"{INPUT_DIR}/{base}.npz"
    np.savez(op, **out)
    print(f"    {base}  ({out['Energy_pu'].shape[0]} steps x {out['Energy_pu'].shape[1]} sites)")
    return op

def _slice_flat(paths):     return [_slice_design(p) for p in paths]
def _slice_by_depth(paths): return [_slice_design(p, depth_m=d) for p in paths for d in OCEAN_CURRENT_DEPTHS]
def _slice_coaxial(paths):
    out = []
    for p in paths:
        d = int(float(np.load(p, allow_pickle=True)["Depth_m"]))
        if d in [int(x) for x in OCEAN_CURRENT_DEPTHS]: out.append(_slice_design(p))
        else: print(f"    skip {os.path.basename(p)} (built at {d} m, not in OCEAN_CURRENT_DEPTHS)")
    return out

print("Wind:");    PathWindDesigns    = _slice_flat(PathWindDesigns)
print("Wave:");    PathWaveDesigns    = _slice_flat(PathWaveDesigns)
print("Kite:");    PathKiteDesigns    = _slice_by_depth(PathKiteDesigns)
print("Coaxial:"); PathCoaxialDesigns = _slice_coaxial(PathCoaxialDesigns)
print("Tidal:");   PathTidalDesigns   = _slice_flat(PathTidalDesigns)

_all = PathWindDesigns + PathWaveDesigns + PathKiteDesigns + PathCoaxialDesigns + PathTidalDesigns
_tlens = {os.path.basename(p): np.load(p, allow_pickle=True)["Energy_pu"].shape[0] for p in _all}
if len(set(_tlens.values())) > 1:
    print("\n  NOTE: designs have different timestep counts; the solver aligns finer cadences")
    print("  to the coarsest by timestamp (handled as long as series share a start time):")
    for k, v in _tlens.items(): print(f"    {v:>6} steps  {k}")
else:
    print(f"\nAll {len(_all)} sliced design(s) share {next(iter(_tlens.values()), 0)} timesteps.")

Run:    Box40p98m41p3_2016_2018_W2-V1
Region: lat[40.98,41.3] lon[-71.13,-70.83]
Years:  2016-2018   |  current depths: n/a
Run folder -> C:/Users/rmiller9/Documents/East Coast Model/Portfolios/Box40p98m41p3_2016_2018_W2-V1

Wind:
    GenPU_ATB_18MW_2030_Box40p98m41p3_2016_2018  (26304 steps x 411 sites)
    GenPU_ATB_12MW_2030_Box40p98m41p3_2016_2018  (26304 steps x 411 sites)
Wave:
    GenPU_RM3_Box40p98m41p3_2016_2018  (8768 steps x 371 sites)
Kite:
Coaxial:
Tidal:

  NOTE: designs have different timestep counts; the solver aligns finer cadences
  to the coarsest by timestamp (handled as long as series share a start time):
     26304 steps  GenPU_ATB_18MW_2030_Box40p98m41p3_2016_2018.npz
     26304 steps  GenPU_ATB_12MW_2030_Box40p98m41p3_2016_2018.npz
      8768 steps  GenPU_RM3_Box40p98m41p3_2016_2018.npz


In [5]:
# ======================================================================
# OPTIMIZATION PARAMETERS
# ======================================================================

LCOE_RANGE = range(120, 90, -6)
Max_CollectionRadious = 30

# Per-technology settings
# MaxDesigns: max number of different designs the optimizer can select
# MinNumTurb: minimum number of devices that must be deployed (0 = optional)
# These are only enforced when the corresponding design list is non-empty.

MaxDesignsWind = 1
MinNumWindTurb = 1

MaxDesingsWave = 1
MinNumWaveTurb = 1

MaxDesingsKite = 1
MinNumKiteTrub = 0

MaxDesignsCoaxial = 1
MinNumCoaxialTurb = 0

MaxDesignsTidal = 1
MinNumTidalTurb = 0

# Turbines/devices per site (controls max density at each location)
# These multiply with NumberOfCellsPerSite in the data files to set upper bounds.
WindTurbinesPerSite = 4       # MW/km² density
WaveTurbinesPerSite = 300     # devices per site cell
KiteTurbinesPerSite = 390     # devices per site cell
CoaxialTurbinesPerSite = 390  # devices per site cell (same grid as kites - HYCOM)
TidalTurbinesPerSite = 200    # devices per site cell (shallower, denser deployments)

## Environmental Exclusion Zones

Toggle individual exclusion layers on/off. When enabled, sites overlapping these zones will be constrained to zero deployment (Y=0) in the optimizer. Set a path to `None` to disable that layer.

In [6]:
# ======================================================================
# ENVIRONMENTAL EXCLUSION ZONES — toggle on/off
# ======================================================================
# Set to True to enable each exclusion layer, False to disable.
# When enabled, the file path must point to the corresponding data file.
# Sites overlapping these zones will be constrained to Y=0 in the optimizer.

ENABLE_HAPC_EXCLUSION = False               # HAPC Hard Bottom Habitat (seabed-sensitive areas)
ENABLE_PROHIBITED_MPA_EXCLUSION = True      # Prohibited Marine Protected Areas (no-go zones)
ENABLE_RESTRICTED_MPA_EXCLUSION = False      # Restricted De Facto MPAs (limited activity zones)

# File paths — set to None to disable regardless of toggle above
HAPC_PATH = "C:/Users/rmiller9/Documents/East Coast Model/HAPC_hard_bottom_habitat.txt" if ENABLE_HAPC_EXCLUSION else None
PROHIBITED_MPA_PATH = "C:/Users/rmiller9/Documents/East Coast Model/prohibitedmpas.txt" if ENABLE_PROHIBITED_MPA_EXCLUSION else None
RESTRICTED_MPA_PATH = "C:/Users/rmiller9/Documents/East Coast Model/restricteddfmpas.txt" if ENABLE_RESTRICTED_MPA_EXCLUSION else None

print("Environmental Exclusion Zones:")
print(f"  HAPC Hard Bottom:        {'ENABLED' if HAPC_PATH else 'DISABLED'}")
print(f"  Prohibited MPAs:         {'ENABLED' if PROHIBITED_MPA_PATH else 'DISABLED'}")
print(f"  Restricted De Facto MPAs: {'ENABLED' if RESTRICTED_MPA_PATH else 'DISABLED'}")

Environmental Exclusion Zones:
  HAPC Hard Bottom:        DISABLED
  Prohibited MPAs:         ENABLED
  Restricted De Facto MPAs: DISABLED


## Shipping Traffic Exclusion Zones

Toggle shipping lane exclusions on/off. When enabled, sites overlapping high-traffic shipping lanes will be constrained to zero deployment.

In [7]:
# ======================================================================
# SHIPPING TRAFFIC EXCLUSION ZONES — toggle on/off
# ======================================================================
# Set to True to enable shipping lane exclusions, False to disable.
# Sites overlapping shipping no-development zones will be constrained to Y=0.

ENABLE_SHIPPING_EXCLUSION = False         # Shipping traffic no-development zones

# File path — set to None to disable regardless of toggle above
SHIPPING_PATH = "C:/Users/rmiller9/Documents/East Coast Model/shipping_no_development.txt" if ENABLE_SHIPPING_EXCLUSION else None

print("Shipping Traffic Exclusion Zones:")
print(f"  Shipping No-Development: {'ENABLED' if SHIPPING_PATH else 'DISABLED'}")

Shipping Traffic Exclusion Zones:
  Shipping No-Development: DISABLED


Pre-Run Summary

In [8]:
print("=" * 70)
print("PRE-RUN SUMMARY")
print("=" * 70)

def _print_tech_summary(tech_name, path_list, devices_per_site):
    if len(path_list) > 0:
        print(f"\n  {tech_name.upper()} ({len(path_list)} design(s))")
        for i, p in enumerate(path_list):
            D = np.load(p, allow_pickle=True)
            n_sites = len(D["LatLong"])
            n_time  = D["Energy_pu"].shape[0]
            rated   = float(D["RatedPower"])
            res_deg = float(D["ResolutionDegrees"])
            res_km  = float(D["ResolutionKm"])
            cells   = D["NumberOfCellsPerSite"]
            lat     = D["LatLong"][:, 0]
            print(f"    [{i}] {os.path.basename(p)}")
            print(f"        Sites: {n_sites:,}  |  Timesteps: {n_time:,}")
            print(f"        Rated Power: {rated} MW  |  Resolution: {res_deg}° ({res_km} km)")
            print(f"        DevicesPerSite setting: {devices_per_site}")
            print(f"        Latitude range: [{lat.min():.2f}, {lat.max():.2f}]")
            D.close()
    else:
        print(f"\n  {tech_name.upper()}: not included")

_print_tech_summary("Wind", PathWindDesigns, WindTurbinesPerSite)
_print_tech_summary("Wave", PathWaveDesigns, WaveTurbinesPerSite)
_print_tech_summary("Kite", PathKiteDesigns, KiteTurbinesPerSite)
_print_tech_summary("Coaxial", PathCoaxialDesigns, CoaxialTurbinesPerSite)
_print_tech_summary("Tidal", PathTidalDesigns, TidalTurbinesPerSite)

print(f"\n  TRANSMISSION ({len(PathTransmissionDesign)} design(s))")
for i, p in enumerate(PathTransmissionDesign):
    print(f"    [{i}] {os.path.basename(p)}")

print("\n" + "=" * 70)

PRE-RUN SUMMARY

  WIND (2 design(s))
    [0] GenPU_ATB_18MW_2030_Box40p98m41p3_2016_2018.npz
        Sites: 411  |  Timesteps: 26,304
        Rated Power: 18.0 MW  |  Resolution: 0.01° (2.0 km)
        DevicesPerSite setting: 4
        Latitude range: [40.98, 41.30]
    [1] GenPU_ATB_12MW_2030_Box40p98m41p3_2016_2018.npz
        Sites: 411  |  Timesteps: 26,304
        Rated Power: 12.0 MW  |  Resolution: 0.01° (2.0 km)
        DevicesPerSite setting: 4
        Latitude range: [40.98, 41.30]

  WAVE (1 design(s))
    [0] GenPU_RM3_Box40p98m41p3_2016_2018.npz
        Sites: 371  |  Timesteps: 8,768
        Rated Power: 0.286 MW  |  Resolution: 0.01° (-1.0 km)
        DevicesPerSite setting: 300
        Latitude range: [40.98, 41.30]

  KITE: not included

  COAXIAL: not included

  TIDAL: not included

  TRANSMISSION (1 design(s))
    [0] Transmission_1200MW_East Coast.npz



In [9]:
# ======================================================================
# RUN OPTIMIZATION — works for any combination of technologies
# ======================================================================

def _extract_name(path):
    """Extract a short case name from a file path."""
    return path.rsplit(r"/")[-1][:-4]

tech_labels = []
if len(PathWindDesigns) > 0:
    tech_labels += [_extract_name(p) for p in PathWindDesigns]
if len(PathWaveDesigns) > 0:
    tech_labels += [_extract_name(p) for p in PathWaveDesigns]
if len(PathKiteDesigns) > 0:
    tech_labels += [_extract_name(p) for p in PathKiteDesigns]
if len(PathCoaxialDesigns) > 0:
    tech_labels += [_extract_name(p) for p in PathCoaxialDesigns]
if len(PathTidalDesigns) > 0:
    tech_labels += [_extract_name(p) for p in PathTidalDesigns]

if len(tech_labels) == 0:
    raise ValueError("At least one technology must have design paths!")

TechCaseName = "_".join(tech_labels)

for PathTransmissionDesign_i in PathTransmissionDesign:
    TransmissionCaseName = _extract_name(PathTransmissionDesign_i)
    
    # outputs live inside the run's folder, alongside its sliced inputs
    PerLCOE_OutputFolder = RUN_DIR + "/" + TransmissionCaseName
    SavePath = RUN_DIR + "/" + TransmissionCaseName + "_summary"
    ReadMe = f"Techs: {TechCaseName} | Transmission: {TransmissionCaseName}"

    print("\n" + "=" * 70)
    print(f"Running: {TechCaseName}")
    print(f"Transmission: {TransmissionCaseName}")
    print(f"Wind: {len(PathWindDesigns)}, Wave: {len(PathWaveDesigns)}, Kite: {len(PathKiteDesigns)}, Coaxial: {len(PathCoaxialDesigns)}, Tidal: {len(PathTidalDesigns)}")
    print(f"Environmental Exclusions: HAPC={'ON' if HAPC_PATH else 'OFF'}, Prohibited={'ON' if PROHIBITED_MPA_PATH else 'OFF'}, Restricted={'ON' if RESTRICTED_MPA_PATH else 'OFF'}")
    print(f"Shipping Exclusions: {'ON' if SHIPPING_PATH else 'OFF'}")
    print("=" * 70 + "\n")

    SolvePortOpt_MaxGen_LCOE_Iterator(
        PathWindDesigns,
        PathWaveDesigns,
        PathKiteDesigns,
        PathCoaxialDesigns,
        PathTidalDesigns,
        PathTransmissionDesign_i,
        LCOE_RANGE,
        Max_CollectionRadious,
        MaxDesignsWind,
        MaxDesingsWave,
        MaxDesingsKite,
        MaxDesignsCoaxial,
        MaxDesignsTidal,
        MinNumWindTurb,
        MinNumWaveTurb,
        MinNumKiteTrub,
        MinNumCoaxialTurb,
        MinNumTidalTurb,
        ReadMe,
        SavePath=SavePath,
        PerLCOE_OutputFolder=PerLCOE_OutputFolder,
        WindTurbinesPerSite=WindTurbinesPerSite,
        WaveTurbinesPerSite=WaveTurbinesPerSite,
        KiteTurbinesPerSite=KiteTurbinesPerSite,
        CoaxialTurbinesPerSite=CoaxialTurbinesPerSite,
        TidalTurbinesPerSite=TidalTurbinesPerSite,
        # Environmental exclusion zone paths (None = disabled)
        HAPCExclusionPath=HAPC_PATH,
        ProhibitedMPAExclusionPath=PROHIBITED_MPA_PATH,
        RestrictedMPAExclusionPath=RESTRICTED_MPA_PATH,
        # Shipping exclusion zone path (None = disabled)
        ShippingExclusionPath=SHIPPING_PATH,
    )

    print(f"\nDone with {SavePath}")


Running: GenPU_ATB_18MW_2030_Box40p98m41p3_2016_2018_GenPU_ATB_12MW_2030_Box40p98m41p3_2016_2018_GenPU_RM3_Box40p98m41p3_2016_2018
Transmission: Transmission_1200MW_East Coast
Wind: 2, Wave: 1, Kite: 0, Coaxial: 0, Tidal: 0
Environmental Exclusions: HAPC=OFF, Prohibited=ON, Restricted=OFF
Shipping Exclusions: OFF

Cost-scaling tag for outputs: '(all 1.0 - no tag)'
Per-LCOE output folder: C:/Users/rmiller9/Documents/East Coast Model/Portfolios/Box40p98m41p3_2016_2018_W2-V1/Transmission_1200MW_East Coast
  Aligning Wind: 26304 -> 8768 timesteps by timestamp
  All technologies aligned to 8768 timesteps
Finding Overlap Site Locations for Wind-Wind
  Wind-Wind: 2303 overlapping pairs found
Finding Overlap Site Locations for Wave-Wave
  Wave-Wave: 0 overlapping pairs found

SITE EXCLUSION CONSTRAINTS

  Loading environmental exclusion zones...
    Loaded Prohibited MPAs: 564 points
  Environmental: 564 exclusion points
    Wind: 0 / 822 sites excluded (0.0%)
    Wave: 0 / 371 sites excluded

  0%|          | 0/5 [00:00<?, ?it/s]

Running Model With LCOE= 120.00
Set parameter OutputFlag to value 1
Set parameter MIPGap to value 0.02
Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (win64 - Windows Server 2022.0 (20348.2))

CPU model: Intel(R) Xeon(R) Gold 6258R CPU @ 2.70GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 56 physical cores, 112 logical processors, using up to 32 threads

Non-default parameters:
MIPGap  0.02

Optimize a model with 15045 rows, 14423 columns and 14914456 nonzeros
Model fingerprint: 0x5c95163c
Variable types: 8768 continuous, 5655 integer (4462 binary)
Coefficient statistics:
  Matrix range     [2e-04, 7e+07]
  Objective range  [1e+00, 1e+05]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+05]
Presolve removed 1199 rows and 2 columns (presolve time = 6s)...
Presolve removed 1889 rows and 75 columns (presolve time = 10s)...
Presolve removed 1889 rows and 75 columns (presolve time = 16s)...
Presolve removed 1889 rows and 75 columns (presolve time = 21s)...
Presolv

  0%|          | 0/5 [04:57<?, ?it/s]

LCOE  119.6 | Wind   2437.8  Wave      0.0  Kite      0.0  Coaxial      0.0  Tidal      0.0  | Curtail   1465.6  Total    972.3 MW
      Wind     GenPU_ATB_18MW_2030_Box40p98m41p3_2016_2018            226 units across  61 sites
      Wave     GenPU_RM3_Box40p98m41p3_2016_2018                        1 units across   1 sites
  Saved per-LCOE file: C:/Users/rmiller9/Documents/East Coast Model/Portfolios/Box40p98m41p3_2016_2018_W2-V1/Transmission_1200MW_East Coast\LCOE_120\Portfolio_LCOE_120.npz
    Loaded Prohibited MPAs: 564 points


...

 80%|████████  | 4/5 [21:50<04:26, 266.11s/it]

LCOE   90.4 | Wind   1685.0  Wave      0.0  Kite      0.0  Coaxial      0.0  Tidal      0.0  | Curtail    769.5  Total    915.5 MW
      Wind     GenPU_ATB_18MW_2030_Box40p98m41p3_2016_2018            156 units across  39 sites
      Wave     GenPU_RM3_Box40p98m41p3_2016_2018                        1 units across   1 sites
  Saved per-LCOE file: C:/Users/rmiller9/Documents/East Coast Model/Portfolios/Box40p98m41p3_2016_2018_W2-V1/Transmission_1200MW_East Coast\LCOE_96\Portfolio_LCOE_96.npz
    Loaded Prohibited MPAs: 564 points


100%|██████████| 5/5 [22:07<00:00, 265.44s/it]

  Saved all outputs for LCOE_96
Saved combined results: C:/Users/rmiller9/Documents/East Coast Model/Portfolios/Box40p98m41p3_2016_2018_W2-V1/Transmission_1200MW_East Coast\Combined_AllLCOE.npz
Saved Summary.csv


Saved Plot_EfficientFrontier.png
Saved Plot_StackedCosts.png

Done with C:/Users/rmiller9/Documents/East Coast Model/Portfolios/Box40p98m41p3_2016_2018_W2-V1/Transmission_1200MW_East Coast_summary
